# Hybrid RAG Question Answering

This notebook develops and evaluates a Retrieval-Augmented Generation (RAG) system for question answering over the screenplay of *12 Angry Men*.

The retrieval pipeline combines two complementary approaches:

- **BM25** for lexical keyword-based retrieval
- **FAISS** with sentence embeddings for semantic retrieval

The retrieved results are combined using **weighted Reciprocal Rank Fusion (RRF)** and provided as context to an instruction-tuned language model for answer generation.

The system is evaluated against the same LLM without retrieval, allowing the contribution of the RAG pipeline to be measured directly.


## Evaluation Dataset

The evaluation dataset contains questions about *12 Angry Men* together with their reference answers. It is used later to compare the Hybrid RAG pipeline with the baseline LLM without retrieval.


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("dataset.csv")

display(df.head())

,Question,Correct Answer,Page References,Explanation
0,What type of train passes by the apartment rig...,An el train (elevated train).,26,This requires retrieving a specific environmen...
1,"According to the boy's alibi, where did he cla...",He claimed to be at the movies.,25,Retrieval of a specific plot point established...
2,What does the sign on the first door of the co...,Court of General Sessions. Part I,3,Fact explicitly stated in the script in the de...
3,What pattern does Juror #3 draw on the same sh...,tic-tac-toe pattern,64,Fact explicitly stated in the script. The info...
4,Which Juror says that he's lived in a slum all...,Juror #5,31,Fact explicitly stated in the script on page 31


## Data Ingestion & Document Chunking

The screenplay is downloaded as a PDF and converted into textual chunks for retrieval.

The document is split at page level, producing a collection of text passages that can be indexed by both lexical and semantic retrieval methods. Chunking reduces the amount of irrelevant context passed to the language model and enables the retriever to identify focused passages related to each question.


In [ ]:
import requests
import fitz  # PyMuPDF

# 1. Download the movie script PDF
url = "https://f004.backblazeb2.com/file/screenplays/posts/12-angry-men-1957/scripts/12%20Angry%20Men%20-%20Release.pdf"
pdf_path = "12_angry_men.pdf"

print("Downloading the script PDF...")
response = requests.get(url)
with open(pdf_path, "wb") as f:
    f.write(response.content)
print("Download complete!")

# 2. Extract text and chunk by page
print("Extracting text from PDF by page...")
doc = fitz.open(pdf_path)

chunks = []

for page_num, page in enumerate(doc):
    page_text = page.get_text().strip()

    # We only add the page if it actually contains text (skips blank pages)
    if page_text:
        # Pro-tip: Adding the page number to the chunk helps the LLM cite its sources!
        chunk_content = f"--- Page {page_num + 1} ---\n{page_text}"
        chunks.append(chunk_content)

print(f"Created {len(chunks)} chunks from {len(doc)} pages.")

# Let's peek at an example chunk
print("\n--- Example Chunk [10] (Page 11) Preview ---")
print(chunks[10][:500]) # Printing the first 500 characters of the 11th chunk
print("--------------------------------------------")

Download complete!
Extracting text from PDF by page...
Created 149 chunks from 149 pages.

--- Example Chunk [10] (Page 11) Preview ---
--- Page 11 ---
11.
#3
(smiling)
I didn't get a chance to look at the
papers today.
#4
I was just wondering how the market
closed.
#3
(pleasantly)
I wouldn't knew. Say, are you on the
exchange or something.
#4
I'm a broker.
#3
Well that's very interesting.
Listen, maybe you can answer a
question for me. I have an uncle
who’s been playing around with some
Canadian stuff...
The foreman turns around, and, as if it is an effort, calls 
out loudly to the others.
FOREMAN
All right, gentlemen. Let’s ta
--------------------------------------------


## Lexical and Semantic Retrieval Indexes

Two complementary retrieval indexes are constructed over the screenplay chunks.

### BM25 Lexical Retrieval

BM25 represents the sparse retrieval component. It scores chunks according to lexical overlap with the query, making it effective when relevant passages contain the same or closely related terms.

### FAISS Semantic Retrieval

For semantic retrieval, each chunk is encoded using the `all-MiniLM-L6-v2` Sentence Transformer model. The resulting dense vectors are indexed with FAISS, enabling retrieval based on semantic similarity rather than exact word overlap.

Using both representations allows the system to combine lexical precision with semantic matching.


In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Building Keyword Database (BM25)...")

# Tokenize 'chunks' (simple space splitting)
tokenized_chunks = [chunk.split( ) for chunk in chunks]

# Initialize the BM25Okapi object
bm25 = BM25Okapi(tokenized_chunks)

print("Building Vector Database (FAISS)...")

# Load the 'all-MiniLM-L6-v2' embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Convert the text chunks into vector embeddings
# Note: SentenceTransformer's encode method returns a NumPy array of float32 by default.
embeddings = embedder.encode(chunks)

# Get the dimension of the embeddings (number of columns in the array)
dimension = embeddings.shape[1]

# Create a FAISS IndexFlatL2 index using the dimension
vector_db = faiss.IndexFlatL2(dimension)

# Add the numpy array of embeddings to the FAISS index
vector_db.add(np.array(embeddings))

Building Keyword Database (BM25)...
Building Vector Database (FAISS)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
chunk_index = 10

print("THE ORIGINAL TEXT (Excerpt)")
# Print only the first 100 characters for brevity
print(chunks[chunk_index][:100], "...\n")


print("HOW BM25 STORES IT")
# The rank_bm25 library stores the word frequency for each chunk in the doc_freqs list.
# The results are kept in the doc_freqs list.
bm25_representation = bm25.doc_freqs[chunk_index]

# Print 10 random keywords and how many times they appeared on this page
print("Sample from the frequency dictionary (Word : Count):")
print(list(bm25_representation.items())[:10], "\n")


print("HOW FAISS STORES IT (VECTOR DB)")
# The NumPy array 'embeddings' has one row for each chunk.
vector_representation = embeddings[chunk_index]

print(f"Total vector size: {len(vector_representation)} dimensions (numbers)")
print("The first 10 numbers of the vector for this chunk:")
print(vector_representation[:10])

ΤΟ ΑΡΧΙΚΟ ΚΕΙΜΕΝΟ (Απόσπασμα)
--- Page 11 ---
11.
#3
(smiling)
I didn't get a chance to look at the
papers today.
#4
I was just wo ...

ΠΩΣ ΤΟ ΕΧΕΙ ΑΠΟΘΗΚΕΥΣΕΙ ΤΟ BM25
Δείγμα από το λεξικό συχνοτήτων (Λέξη : Φορές):
[('---', 2), ('Page', 1), ('11', 1), ('11.', 1), ('#3', 4), ('(smiling)', 1), ('I', 5), ("didn't", 1), ('get', 2), ('a', 7)] 

ΠΩΣ ΤΟ ΕΧΕΙ ΑΠΟΘΗΚΕΥΣΕΙ ΤΟ FAISS (VECTOR DB)
Συνολικό μέγεθος διανύσματος: 384 διαστάσεις (αριθμοί)
Οι πρώτοι 10 αριθμοί του διανύσματος για αυτό το chunk:
[-0.044679    0.03584594 -0.00903327 -0.03461855 -0.01736854  0.0230823
  0.12162165  0.03595631  0.08935086  0.09200873]


## Hybrid Retrieval with Reciprocal Rank Fusion

The hybrid retriever searches both the BM25 lexical index and the FAISS semantic index for each query.

The two ranked result lists are combined using weighted Reciprocal Rank Fusion (RRF). This allows evidence retrieved by either lexical or semantic similarity to contribute to the final ranking while avoiding direct comparison between the different scoring scales used by BM25 and FAISS.

The highest-ranked chunks are returned as context for answer generation.


In [ ]:
def hybrid_retrieve(query, top_k=2):
    """
    Retrieves top chunks using BM25 and Vector Search, then combines the results.
    """

    candidate_k = 8

    # Keyword search
    tokenized_query = query.split()

    # BM25 scoring
    bm25_scores = bm25.get_scores(tokenized_query)

    # Top BM25 matches
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:candidate_k].tolist()


    # Semantic search
    query_embedding = embedder.encode([query])


    # FAISS retrieval
    distances, faiss_top_indices = vector_db.search(
        np.array(query_embedding).astype("float32"),
        candidate_k
    )
    faiss_top_indices = faiss_top_indices[0].tolist()

    # Weighted Reciprocal Rank Fusion
    k = 60
    rrf_scores = {}

    bm25_weight = 2.0
    faiss_weight = 1.0

    for rank, index in enumerate(bm25_top_indices):
        rrf_scores[index] = rrf_scores.get(index, 0) + bm25_weight / (k + rank + 1)

    for rank, index in enumerate(faiss_top_indices):
        rrf_scores[index] = rrf_scores.get(index, 0) + faiss_weight / (k + rank + 1)

    final_combined_indices = sorted(
        rrf_scores.keys(),
        key=lambda index: rrf_scores[index],
        reverse=True
    )[:top_k]

    # Retrieve the final context chunks
    retrieved_chunks = [chunks[index] for index in final_combined_indices]

    return retrieved_chunks

test_query = "Who had a cold?"



Found 2 relevant chunks for the test query.

--- Chunk 1 ---
--- Page 10 ---
10.
Yeah! And what about the business
with the knife- I mean asking grown-
up people to believe that kind of
bushwash.
#8 turns during these lines to look at #7.
MEDIUM SHOT - #7 AND #10
#10 sits in a chair not at table, #7 stands over him mopping 
his brow
#10
Well look, you've gotta expect that.
You know what you’re dealing with.
#7
Yeah, I suppose.
#10 blows his nose vigorously.
#7
What’s the matter, you got a cold?
#10
And how. These hot weather colds can
kill you.
(he tilts his head back 
slightly)
I can hardly touch my nose. Know
what I mean?
#7 nods sympathetically.
#7
I just got over one.
There is an awkward pause. #7 looks at his watch. Then he 
looks up at Foreman, who is standing at head of table.
#7
What d’ya say, Mr. Foreman?
MEDIUM SHOT - #7, #10, FOREMAN, #3, #4, FROM ANOTHER ANGLE
Foreman big in foreground, standing at head of table. #7, #10 
in background. #3, #4 seated at left at table. Forem

## Language Model


Answer generation uses `Qwen3-4B-Instruct-2507`. The model receives either retrieved screenplay context in the RAG configuration or only the question in the baseline configuration.

In [ ]:
import torch
from transformers import pipeline

print("Loading LLM...")

model_id = "Qwen/Qwen3-4B-Instruct-2507"

# Set up the text-generation pipeline
llm_pipeline = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"dtype": torch.bfloat16},
    device_map="auto"
)

print("LLM Loaded and ready!")

Loading LLM...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

LLM Loaded and ready!


## RAG Question Answering Pipeline

The complete RAG pipeline combines retrieval and generation.

For each question, the hybrid retriever selects the most relevant screenplay chunks. These passages are combined into a structured context and supplied to the language model together with instructions to answer using only the retrieved evidence.

This grounding step is designed to reduce unsupported answers and make the generated response dependent on evidence retrieved from the screenplay.


In [ ]:
def ask_rag(query):
    """
    Executes the full RAG pipeline: Retrieves context, builds prompt, and generates an answer.
    """
    print(f"Question: {query}")
    print("Retrieving context...")
    contexts = hybrid_retrieve(query, top_k=3)

    # Join the retrieved chunks into a single string (e.g., separated by newlines)
    context_string = "\n\n---\n\n".join(contexts)

    # Build the prompt
    # Write a strong system prompt directing the AI's behavior
    system_prompt = """ You are an expert assistant analyzing the movie "12 Angry Men".
                        Answer the user's question using ONLY the provided context from the movie script.
                        Do not use outside knowledge.
                        If the answer is not clearly supported by the context, say that the context does not provide enough information.
                        Keep the answer concise and factual.
                        When possible, mention the page or chunk evidence from the provided context.
                    """

    # Combine the context and the user's query / Change this as needed to fit your system prompt and the way you want to present the information to the LLM.
    user_prompt = f"Context:\n{context_string}\n\nQuestion: {query}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # Generate Answer / You can play with the generation parameters to see how they affect the output. For factual answers, lower temperature and disabling sampling can help.
    outputs = llm_pipeline(
        messages,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=True
    )

    # Extract just the text of the assistant's reply
    final_answer = outputs[0]["generated_text"][-1]["content"]

    return final_answer

## Baseline Comparison

To measure the contribution of retrieval, the same language model is also evaluated without access to retrieved screenplay context.

The baseline receives only the question and therefore relies on information encoded in the model itself. Comparing the two configurations isolates the effect of retrieval on question-answering performance.


In [ ]:
def ask_baseline(query):
    """
    Queries the LLM directly without any retrieved context (Baseline).
    """
    print(f"Question: {query}")
    print("Generating baseline answer...")

    # Build the prompt without context
    system_prompt = "You are a helpful assistant. Answer the user's question to the best of your ability. The question will be about the movie '12 Angry men' of 1957."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]

    # Generate Answer
    outputs = llm_pipeline(
        messages,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=False
    )

    # Extract just the text of the assistant's reply
    final_answer = outputs[0]["generated_text"][-1]["content"]
    return final_answer

In [ ]:
ask_rag("whats the name of the character who has a cold?")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: whats the name of the character who has a cold?
Retrieving context...


'The character who has a cold is Juror #10.  \n\nEvidence from Page 2: "Juror #10: 46 years old. Garage owner. An angry, bitter man... He has a bad cold and continually blows his nose, sniffs a benzedrine inhaler, etc."  \n\nAlso, on Page 10: "#10 blows his nose vigorously." This confirms Juror #10 has a cold.'

In [ ]:
ask_baseline("whats the name of the character who has a cold?")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: whats the name of the character who has a cold?
Generating baseline answer...


'In the 1957 film *12 Angry Men*, there is no character explicitly mentioned as having a cold. The film focuses on a group of 12 jurors who must decide the fate of a young man accused of murdering his father. While the characters have various personal backgrounds, motivations, and emotions, none of them are described as having a cold or any physical illness.\n\nSo, to answer your question: **There is no character in *12 Angry Men* who is named or described as having a cold.**'

## Evaluation

The Hybrid RAG system and the baseline LLM are evaluated on the same set of 10 questions about *12 Angry Men*.

Generated answers are compared with the reference answers in the evaluation dataset. Using the same language model in both configurations makes it possible to examine how access to retrieved evidence affects answer accuracy.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# EVALUATION SETTINGS
# ==========================================
# Set this to False if you only want to generate the CSV without printing to the screen
PRINT_RESULTS = True

# Define how many questions to test (e.g., first 5, 10, or len(df_questions) for all)
num_tests = 10
# ==========================================

print("Loading dataset...")
# Load your local dataset
file_path = "dataset.csv"  # Ensure this file is in the same folder as your notebook
df_questions = pd.read_csv(file_path)

print(f"Loaded {len(df_questions)} questions from {file_path}")
if PRINT_RESULTS:
    display(df_questions.head())

# 1. Initialize the new columns in the dataframe if they don't exist
if "simple_rag_answer" not in df_questions.columns:
    df_questions["simple_rag_answer"] = ""
if "baseline_answer" not in df_questions.columns:
    df_questions["baseline_answer"] = ""

# Adjust num_tests just in case the dataset is smaller than requested
num_tests = min(num_tests, len(df_questions))
print(f"\nEvaluating {num_tests} questions. This may take a few minutes...\n")

for index, row in df_questions.head(num_tests).iterrows():
    question = row["Question"]
    expected = row["Correct Answer"]

    # 2. Get answers from both systems for comparison
    baseline_model_answer = ask_baseline(question)
    rag_model_answer = ask_rag(question)

    # 3. Save the answers into the dataframe
    df_questions.at[index, "baseline_answer"] = baseline_model_answer
    df_questions.at[index, "simple_rag_answer"] = rag_model_answer

    # 4. Pretty print the results for real-time monitoring
    if PRINT_RESULTS:
        output = f"""
### Question {index + 1}: {question}

**Expected Answer:**
> {expected}

**Baseline Model Answer (No Context):**
> {baseline_model_answer}

**RAG Model Answer (With Context):**
> {rag_model_answer}

---
"""
        display(Markdown(output))

# 5. Save the updated dataframe to your local CSV
df_questions.to_csv("dataset_with_results.csv", index=False)
print(f"✅ Successfully processed {num_tests} questions and saved to 'dataset_with_results.csv'")

Loading dataset...
Loaded 26 questions from dataset.csv


,Question,Correct Answer,Page References,Explanation
0,What type of train passes by the apartment rig...,An el train (elevated train).,26,This requires retrieving a specific environmen...
1,"According to the boy's alibi, where did he cla...",He claimed to be at the movies.,25,Retrieval of a specific plot point established...
2,What does the sign on the first door of the co...,Court of General Sessions. Part I,3,Fact explicitly stated in the script in the de...
3,What pattern does Juror #3 draw on the same sh...,tic-tac-toe pattern,64,Fact explicitly stated in the script. The info...
4,Which Juror says that he's lived in a slum all...,Juror #5,31,Fact explicitly stated in the script on page 31


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Evaluating 10 questions. This may take a few minutes...

Question: What type of train passes by the apartment right outside the window where the murder took place?
Generating baseline answer...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Question: What weapon was used in the murder?
Generating baseline answer...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What weapon was used in the murder?
Retrieving context...



### Question 10: What weapon was used in the murder?

**Expected Answer:**
> A knife

**Baseline Model Answer (No Context):**
> In the 1957 film *12 Angry Men*, the weapon used in the murder is a **rope**. The victim, a young man, was strangled with a rope. This detail is revealed during the course of the trial as the jury members discuss the evidence and the circumstances of the crime. The rope is a key piece of evidence that the defendant, who is accused of the murder, is said to have used. However, the film emphasizes the importance of the jury's deliberation and the process of reaching a fair verdict rather than focusing on the specifics of the weapon itself.

**RAG Model Answer (With Context):**
> The weapon used in the murder was a switch-knife. 

Evidence from the context:  
On page 122, #5 states, "Like that. Anyone who's ever used a switch-knife'd never handle it any other way." Additionally, #5 says, "He'd go for him underhanded..." and "No, I don’t think he could. He'd go for him underhanded..." This indicates the knife was a switch-knife, and the manner of use is described. The knife is also referenced as the murder weapon when #8 asks, "Do you think he could have made the kind of wound that killed his father?"  

Thus, the weapon used in the murder was a

---


✅ Successfully processed 10 questions and saved to 'dataset_with_results.csv'


## Results & Error Analysis

The Hybrid RAG pipeline answered **8 out of 10 questions correctly**, while the baseline LLM answered **1 out of 10 correctly**.

### RAG Failure Analysis

For the two questions answered incorrectly by the RAG system, the generated responses indicated that the retrieved context did not contain enough information to answer the question. This suggests that the primary failure occurred during retrieval rather than answer generation: the relevant screenplay evidence was not included in the context supplied to the LLM.

Possible sources of this limitation include the page-level chunking strategy and the retrieval representations used to rank candidate passages.

### Baseline Failure Analysis

Without retrieved screenplay evidence, the baseline model frequently produced unsupported or incorrect answers. In several cases it generated plausible-sounding responses despite lacking the information required to answer the question accurately.

The comparison demonstrates the importance of grounding the language model in retrieved evidence for this question-answering task.
